# Urdu Diffusion-BERT on Kaggle (with Google Drive checkpoints)

Pretrain a small text-diffusion BERT (MDLM) on Urdu, evaluate on Urdu sentiment + XNLI using **instruction-prompted diffusion generation**, then back up checkpoints + results to **Google Drive** so they survive Kaggle's 9-hour session limit.

---

## One-time setup (BEFORE opening Kaggle)

**A. Google Cloud + Drive credentials**
1. https://console.cloud.google.com/ → create / pick a project.
2. Enable **Google Drive API**.
3. APIs & Services → Credentials → **Create credentials → OAuth client ID → Desktop app** → download `credentials.json`.
4. On your laptop: `python setup_gdrive.py` → opens browser → sign in → produces `token.pickle`.
5. In Google Drive, create a folder (e.g. `ur_mdlm_ckpts`). Copy its ID from the URL `https://drive.google.com/drive/folders/<THIS_PART>`.

**B. Upload `token.pickle` as a private Kaggle Dataset**
1. https://www.kaggle.com/datasets → **New Dataset** → drag `token.pickle` in → **Private** → Create.
2. Note the dataset slug, e.g. `yourname/gdrive-token`.

**C. Kaggle notebook settings**
1. Settings → Accelerator → **GPU T4 x2** (or P100).
2. Settings → Internet → **On**.
3. Add data → search for your `gdrive-token` dataset → **Attach**. It will mount at `/kaggle/input/<dataset-slug>/token.pickle`.

## Cell 1 — Clone the repo

In [ ]:
!git clone https://github.com/sushanedulloo/Diffusion-Bert.git /kaggle/working/bert
%cd /kaggle/working/bert
!git log -1 --oneline

## Cell 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers sentencepiece \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib

## Cell 3 — Configure paths (EDIT THESE!)

In [ ]:
import os, glob

# === EDIT THESE TWO LINES ===
GDRIVE_TOKEN     = "/kaggle/input/gdrive-token/token.pickle"   # path inside the attached dataset
GDRIVE_FOLDER_ID = "PASTE_YOUR_DRIVE_FOLDER_ID_HERE"           # ur_mdlm_ckpts folder ID
# ============================

OUT_DIR     = "/kaggle/working/ur_mdlm_mini"
RESULTS_DIR = "/kaggle/working/results"

assert os.path.exists(GDRIVE_TOKEN), (
    f"token.pickle not found at {GDRIVE_TOKEN}.\n"
    "Did you attach the gdrive-token Kaggle Dataset under 'Add data'?\n"
    "Available .pickle under /kaggle/input/: "
    + str(glob.glob('/kaggle/input/**/*.pickle', recursive=True))
)
assert GDRIVE_FOLDER_ID != "PASTE_YOUR_DRIVE_FOLDER_ID_HERE", "Set GDRIVE_FOLDER_ID first."
print("Paths OK.\n  Token :", GDRIVE_TOKEN, "\n  Folder:", GDRIVE_FOLDER_ID)

## Cell 4 — Hugging Face login (for gated IndicCorp v2)

Paste your HF token. Then in a browser, sign in to https://huggingface.co/datasets/ai4bharat/IndicCorpV2 and click **"Agree and access repository"** once.

**Don't want IndicCorp?** Skip this cell and add `--no_indiccorp` to Cell 5 — it will use Urdu Wikipedia alone.

In [ ]:
from huggingface_hub import login
login(token="hf_xxx_paste_your_token_here")

## Cell 5 — Pretrain Urdu MDLM (local checkpoints)

We deliberately do **not** pass `--gdrive_folder_id` to `train.py`, because that flag deletes the local checkpoint right after upload — and we need a local copy for the eval cells. We'll upload at the end (Cell 10) instead.

**OOM?** Use `--batch_size 16 --gradient_accumulation_steps 8`.
**Time to spare?** Raise `--max_docs 100000` → `500000`, or switch `--model_size mini` → `small`.

In [ ]:
!python train.py \
    --language ur \
    --training_objective mdlm --noise_schedule cosine \
    --model_size mini \
    --tokenizer_name jhu-clsp/mmBERT-base \
    --max_docs 100000 \
    --num_epochs 1 \
    --batch_size 32 --gradient_accumulation_steps 4 \
    --phase1_max_seq_len 128 --phase2_max_seq_len 128 \
    --learning_rate 2e-4 \
    --dataloader_num_workers 2 \
    --output_dir {OUT_DIR}

## Cell 6 — Locate the final checkpoint

Checkpoints live at `<OUT_DIR>/checkpoint_step_NNNNNNN/checkpoint.pt` and `<OUT_DIR>/checkpoint_final/checkpoint.pt`. `evaluate.py --checkpoint <dir>` appends `checkpoint.pt` to whatever dir you pass — so we point it at the sub-folder, not `OUT_DIR` itself.

In [ ]:
import os, glob
for d in sorted(glob.glob(os.path.join(OUT_DIR, 'checkpoint_*'))):
    pt = os.path.join(d, 'checkpoint.pt')
    mb = os.path.getsize(pt) / 1e6 if os.path.exists(pt) else 0
    print(f'{d}   ({mb:.1f} MB)')

CKPT_DIR = os.path.join(OUT_DIR, 'checkpoint_final')
assert os.path.exists(os.path.join(CKPT_DIR, 'checkpoint.pt')), \
    f'Final checkpoint missing at {CKPT_DIR}/checkpoint.pt — did training finish?'
print('\nCKPT_DIR =', CKPT_DIR)

## Cell 7 — Evaluate on Urdu sentiment (IndicSentiment)

Prompt format:
```
[CLS] اس جائزے کا جذبہ کیا ہے؟ جائزہ: <text> جذبہ: [MASK][MASK] [SEP]
```
The model has to fill the `[MASK]` slots with `مثبت` (positive) or `منفی` (negative).

In [ ]:
!python evaluate.py urdu_sentiment \
    --checkpoint {CKPT_DIR} \
    --output_dir {RESULTS_DIR} \
    --tokenizer_name jhu-clsp/mmBERT-base \
    --epochs 5 \
    --batch_size 16 \
    --num_steps 50

## Cell 8 — Evaluate on Urdu XNLI

Capped to 20k train / 2k eval to fit within a Kaggle session. Drop `--max_train`/`--max_eval` for the full run.

In [ ]:
!python evaluate.py urdu_nli \
    --checkpoint {CKPT_DIR} \
    --output_dir {RESULTS_DIR} \
    --tokenizer_name jhu-clsp/mmBERT-base \
    --max_train 20000 --max_eval 2000 \
    --epochs 2 \
    --batch_size 16 \
    --num_steps 50

## Cell 9 — Read results

In [ ]:
import json, os
for f in sorted(os.listdir(RESULTS_DIR)):
    path = os.path.join(RESULTS_DIR, f)
    print(f)
    print(json.dumps(json.load(open(path)), indent=2, ensure_ascii=False))
    print('---')

## Cell 10 — Back everything up to Google Drive

Uploads:
1. The **final checkpoint** as `latest_checkpoint.pt` (using the repo's `gdrive_uploader.upload_checkpoint`, which overwrites the previous version so you don't accumulate copies).
2. Every **eval results JSON** as an individual file (overwrites if it already exists).

All into the Drive folder whose ID you set in Cell 3.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/bert')
from gdrive_uploader import upload_checkpoint, _get_service
from googleapiclient.http import MediaFileUpload

# 1) Final checkpoint → latest_checkpoint.pt on Drive
upload_checkpoint(
    checkpoint_dir = CKPT_DIR,
    folder_id      = GDRIVE_FOLDER_ID,
    token_path     = GDRIVE_TOKEN,
)
print('Checkpoint uploaded as latest_checkpoint.pt')

# 2) Every results JSON
service = _get_service(GDRIVE_TOKEN)
for fname in sorted(os.listdir(RESULTS_DIR)):
    local = os.path.join(RESULTS_DIR, fname)
    q = f"name='{fname}' and '{GDRIVE_FOLDER_ID}' in parents and trashed=false"
    existing = service.files().list(q=q, fields='files(id)').execute().get('files', [])
    media = MediaFileUpload(local, mimetype='application/json', resumable=False)
    if existing:
        service.files().update(fileId=existing[0]['id'], media_body=media).execute()
        print(f'Overwrote {fname} on Drive')
    else:
        meta = {'name': fname, 'parents': [GDRIVE_FOLDER_ID]}
        service.files().create(body=meta, media_body=media, fields='id').execute()
        print(f'Uploaded {fname} to Drive')

## Cell 11 (optional) — Qualitative samples

Unconditional reverse-diffusion samples from the trained model. Useful to eyeball whether pretraining produced fluent Urdu.

In [ ]:
import torch
from transformers import AutoTokenizer
from evaluate.diffusion_utils import (
    load_diffusion_model_from_checkpoint, resolve_checkpoint, mdlm_sample_conditional,
)
from training.noise_schedule import LinearAlphaScheduler

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('jhu-clsp/mmBERT-base')
model     = load_diffusion_model_from_checkpoint(resolve_checkpoint(CKPT_DIR)).to(device).eval()
scheduler = LinearAlphaScheduler()

B, L = 4, 64
input_ids = torch.full((B, L), tokenizer.mask_token_id, dtype=torch.long, device=device)
input_ids[:, 0]  = tokenizer.cls_token_id
input_ids[:, -1] = tokenizer.sep_token_id
attention_mask = torch.ones_like(input_ids)
condition_mask = torch.zeros_like(input_ids, dtype=torch.bool)
condition_mask[:, 0]  = True
condition_mask[:, -1] = True

generated = mdlm_sample_conditional(
    model=model, tokenizer=tokenizer,
    input_ids=input_ids, condition_mask=condition_mask,
    scheduler=scheduler, attention_mask=attention_mask,
    num_steps=200, temperature=1.0, device=device,
)
for i, ids in enumerate(generated):
    print(f'[{i}]', tokenizer.decode(ids, skip_special_tokens=True))

## Cell 12 (next session) — Resume from Drive

If you come back in a fresh Kaggle session: run Cells 1, 2, 3, then this cell to pull `latest_checkpoint.pt` from Drive into `<OUT_DIR>/checkpoint_final/checkpoint.pt`. Cells 7+ then work as-is.

In [ ]:
import os
from googleapiclient.http import MediaIoBaseDownload
from gdrive_uploader import _get_service

service = _get_service(GDRIVE_TOKEN)
q = f"name='latest_checkpoint.pt' and '{GDRIVE_FOLDER_ID}' in parents and trashed=false"
files = service.files().list(q=q, fields='files(id,name,size)').execute().get('files', [])
assert files, 'latest_checkpoint.pt not found in your Drive folder.'
file_id = files[0]['id']
print(f"Found {files[0]['name']} ({int(files[0].get('size', 0)) / 1e6:.1f} MB) — downloading …")

dst_dir = os.path.join(OUT_DIR, 'checkpoint_final')
os.makedirs(dst_dir, exist_ok=True)
dst = os.path.join(dst_dir, 'checkpoint.pt')

req = service.files().get_media(fileId=file_id)
with open(dst, 'wb') as fh:
    downloader = MediaIoBaseDownload(fh, req, chunksize=16 * 1024 * 1024)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(f"  {int(status.progress() * 100)}%")
print('Saved to', dst)
CKPT_DIR = dst_dir